# Feature Selection Method Based on Grey Wolf Optimization for Coronary Artery Disease Classification

In [1]:
import random
import time
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import benchmarks

data = pd.read_csv("cleavland.csv")
data.head()

c:\Users\asus\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,AGE,SEX,CP,TRESTBPS,CHOL,FBD,REST ECG,THALACH,EXANG,OLDPEAK,SLOPE,CA,THAL,RESULT
0,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0


### Data preprocessing, extraction and splitting

In [2]:
X = data.iloc[:,:13].values
Y = data['RESULT'].values
print(X.shape)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.40, random_state = 14)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

(303, 13)


### Applying SVM classification without data processing

In [3]:
svcclassifier = SVC(kernel = 'poly', random_state = 50)
svcclassifier.fit(X_train, y_train)
y_pred = svcclassifier.predict(X_test)
P = accuracy_score(y_pred,y_test)
print("Accuracy score for SVM:",P)

Accuracy score for SVM: 0.5983606557377049


### GWO Optimization algo

In [4]:
def GWO(objf,lb,ub,dim,SearchAgents_no,Max_iter):
    
    # initialize alpha, beta, and delta_pos
    Alpha_pos=np.zeros(dim)
    Alpha_score=float("inf")
    
    Beta_pos=np.zeros(dim)
    Beta_score=float("inf")
    
    Delta_pos=np.zeros(dim)
    Delta_score=float("inf")

    if not isinstance(lb, list):
        lb = [lb] * dim
    if not isinstance(ub, list):
        ub = [ub] * dim
    
    #Initialize the positions of search agents
    Positions = np.zeros((SearchAgents_no, dim))
    for i in range(dim):
        Positions[:, i] = np.random.uniform(0,1, SearchAgents_no) * (ub[i] - lb[i]) + lb[i]
    
    Convergence_curve=np.zeros(Max_iter)

     # Loop counter
    print("GWO is optimizing  \""+objf.__name__+"\"")    

    # Main loop
    for l in range(0,Max_iter):
        for i in range(0,SearchAgents_no):
            
            # Return back the search agents that go beyond the boundaries of the search space
            for j in range(dim):
                Positions[i,j]=np.clip(Positions[i,j], lb[j], ub[j])

            # Calculate objective function for each search agent
            fitness=objf(Positions[i,:])
            
            # Update Alpha, Beta, and Delta
            if fitness<Alpha_score :
                Alpha_score=fitness; # Update alpha
                Alpha_pos=Positions[i,:].copy()
            
            
            if (fitness>Alpha_score and fitness<Beta_score ):
                Beta_score=fitness  # Update beta
                Beta_pos=Positions[i,:].copy()
            
            
            if (fitness>Alpha_score and fitness>Beta_score and fitness<Delta_score): 
                Delta_score=fitness # Update delta
                Delta_pos=Positions[i,:].copy()
        
        a=2-l*((2)/Max_iter); # a decreases linearly fron 2 to 0
        
        # Update the Position of search agents including omegas
        for i in range(0,SearchAgents_no):
            for j in range (0,dim):     
                           
                r1=random.random() # r1 is a random number in [0,1]
                r2=random.random() # r2 is a random number in [0,1]
                
                A1=2*a*r1-a; # Equation (3.3)
                C1=2*r2; # Equation (3.4)
                
                D_alpha=abs(C1*Alpha_pos[j]-Positions[i,j]); # Equation (3.5)-part 1
                X1=Alpha_pos[j]-A1*D_alpha; # Equation (3.6)-part 1
                           
                r1=random.random()
                r2=random.random()
                
                A2=2*a*r1-a; # Equation (3.3)
                C2=2*r2; # Equation (3.4)
                
                D_beta=abs(C2*Beta_pos[j]-Positions[i,j]); # Equation (3.5)-part 2
                X2=Beta_pos[j]-A2*D_beta; # Equation (3.6)-part 2       
                
                r1=random.random()
                r2=random.random() 
                
                A3=2*a*r1-a; # Equation (3.3)
                C3=2*r2; # Equation (3.4)
                
                D_delta=abs(C3*Delta_pos[j]-Positions[i,j]); # Equation (3.5)-part 3
                X3=Delta_pos[j]-A3*D_delta; # Equation (3.5)-part 3             
                
                Positions[i,j]=(X1+X2+X3)/3  # Equation (3.7)        
        Convergence_curve[l]=Alpha_score;

        #if (l%1==0):
               #print(['At iteration '+ str(l)+ ' the best fitness is '+ str(Alpha_score)]);
    
    print(Positions.shape)
    print("Alpha position=",Alpha_pos);
    print("Beta position=",Beta_pos);
    print("Delta position=",Delta_pos);
    return Alpha_pos,Beta_pos;


### Setting GWO parameters

In [5]:
iters=100
wolves=5
dimension=13
search_domain=[0,1]
lb=-1.28
ub=1.28
colneeded=[0,1,2,4,5,7,8,10,11]
modified_data=pd.DataFrame()
for i in colneeded:
    modified_data[data.columns[i]]=data[data.columns[i]].astype(float)
func_details=benchmarks.getFunctionDetails(6)

for i in range(0,10):
    alpha,beta=GWO(getattr(benchmarks,'F7'),lb,ub,dimension,wolves,iters)

GWO is optimizing  "F7"
(5, 13)
Alpha position= [-0.01779236  0.11658362  0.08580658 -0.09747281 -0.10514813 -0.13390837
  0.02106524  0.0216336   0.08687137 -0.15234753  0.04999521  0.00191522
 -0.18494057]
Beta position= [-0.01705799  0.110957    0.0873013  -0.1044995  -0.13078831 -0.13482537
  0.0195706   0.02417723  0.07678404 -0.15080467  0.05045137  0.00145967
 -0.18950092]
Delta position= [-0.01693238  0.10867137  0.06887691 -0.0863015  -0.08294455 -0.13311015
  0.02019048  0.02617916  0.10378849 -0.1564553   0.05676222  0.00221294
 -0.22001111]
GWO is optimizing  "F7"
(5, 13)
Alpha position= [ 0.21582375 -0.21529695  0.02971087  0.26397038  0.02613939  0.19470166
 -0.24891373  0.08335149 -0.12323252  0.07986403  0.14031279  0.02431483
  0.15755107]
Beta position= [ 0.11645507 -0.26001392  0.03093091  0.25615207  0.04128209  0.21422675
 -0.19451649  0.09060821 -0.10862266  0.04661628  0.11972298  0.02135011
  0.2302717 ]
Delta position= [ 0.1497137  -0.24010311  0.02978979  0.26

In [6]:
##Applying feature selection on the given dataset
##considering alpha as best solution and putting a threshold
threshold=-0.05
index=[]
print("alpha shape=",alpha.shape[0])
modified_daata=pd.DataFrame();
for i in range(0,alpha.shape[0]):
    if(alpha[i]>=threshold):
        modified_daata[data.columns[i]]=data[data.columns[i]].astype(float)
print("The modified data is following")
modified_daata.head()

alpha shape= 13
The modified data is following


,AGE,SEX,CP,TRESTBPS,CHOL,REST ECG,THALACH,EXANG,CA,THAL
0,63.0,1.0,1.0,145.0,233.0,2.0,150.0,0.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,2.0,108.0,1.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,2.0,129.0,1.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,187.0,0.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,2.0,172.0,0.0,0.0,3.0


### Applying svm on the modified data

In [7]:
Y = data['RESULT'].values
X_train, X_test, y_train, y_test = train_test_split(modified_daata, Y, test_size = 0.40, random_state = 14)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
print(X_train[0])
modified_daata.info()

[-0.37086712  0.6520712  -0.1543891  -1.21573705 -1.45429374 -0.9863518
 -1.07775066 -0.76662853 -0.77346436 -0.95367089]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   AGE       303 non-null    float64
 1   SEX       303 non-null    float64
 2   CP        303 non-null    float64
 3   TRESTBPS  303 non-null    float64
 4   CHOL      303 non-null    float64
 5   REST ECG  303 non-null    float64
 6   THALACH   303 non-null    float64
 7   EXANG     303 non-null    float64
 8   CA        303 non-null    float64
 9   THAL      303 non-null    float64
dtypes: float64(10)
memory usage: 23.8 KB


In [8]:
svcclassifier = SVC(kernel = 'rbf', random_state = 50)
svcclassifier.fit(X_train, y_train)
y_pred = svcclassifier.predict(X_test)
P1=accuracy_score(y_pred,y_test)
print("Modified Accuracy score for SVM:", P1)
print("Earlier Accuracy score was", P)

Modified Accuracy score for SVM: 0.6147540983606558
Earlier Accuracy score was 0.5983606557377049
